# SpectraShift Week 8: freeze sealed evaluation
Use CPU with Internet off. Attach source v7, Week 2 frozen, Week 5 contracts, Week 7 complete, and the private evaluation-candidate dataset.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week8.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 8 source bundle, found {bundles}'
    source_work = Path('/tmp/spectrashift-week8-source')
    if source_work.exists(): shutil.rmtree(source_work)
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 8 source tree found'
PROJECT = sorted(projects, key=lambda path: len(str(path)))[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

def unique_file(name):
    candidates = sorted(INPUT.rglob(name))
    by_hash = {}
    for path in candidates:
        by_hash.setdefault(hashlib.sha256(path.read_bytes()).hexdigest(), path)
    assert len(by_hash) == 1, f'Expected one unique {name}; found {candidates}'
    return next(iter(by_hash.values()))

def install_offline_foundation_dependencies():
    wheels = sorted(INPUT.rglob('foundation-wheels'))
    if not wheels: return
    missing = []
    for module, package in [('upath','universal-pathlib'), ('omegaconf','omegaconf'), ('iopath','iopath'), ('fvcore','fvcore'), ('einops','einops'), ('huggingface_hub','huggingface_hub')]:
        try: __import__(module)
        except ImportError: missing.append(package)
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', str(wheels[0]), *missing])

WORK = Path('/kaggle/working/spectrashift-week8-evaluation-contracts')
WORK.mkdir(parents=True, exist_ok=True)
MANIFEST = unique_file('partitions.parquet')
NORMALIZATION = unique_file('normalization.json')
FREEZE = unique_file('freeze_summary.json')
WEEK7_SUMMARY = unique_file('week7_run_summary.json')
LEDGER = unique_file('week8_checkpoint_ledger.csv')
STAGED = next(path.parent for path in INPUT.rglob('staging_summary.json'))
WEEK5_CONTRACTS = unique_file('week5_contracts_summary.json')
CANDIDATES = unique_file('evaluation_candidate_labels.parquet')
config = yaml.safe_load((PROJECT / 'configs/eval/week8.yaml').read_text())
config['paths'].update({
    'manifest_path': str(MANIFEST), 'staged_root': str(STAGED),
    'normalization_path': str(NORMALIZATION), 'freeze_summary_path': str(FREEZE),
    'week5_contracts_path': str(WEEK5_CONTRACTS), 'week7_summary_path': str(WEEK7_SUMMARY),
    'checkpoint_ledger_path': str(LEDGER), 'candidate_labels_path': str(CANDIDATES),
    'output_dir': str(WORK), 'evaluation_labels_path': str(WORK / 'evaluation_labels.parquet'),
    'evaluation_contract_path': str(WORK / 'evaluation_contract.json'),
    'support_contract_path': str(WORK / 'support_contract.json'),
})
RUNTIME_CONFIG = WORK / 'week8.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print({'project': str(PROJECT), 'work': str(WORK), 'candidate_labels': str(CANDIDATES)})


In [ ]:
from spectrashift.train.week8 import freeze_week8_evaluation
summary = freeze_week8_evaluation(RUNTIME_CONFIG)
print(json.dumps(summary, indent=2))
assert summary['week8_sealing_complete']
assert summary['partition_counts'] == {'I': 3000, 'T-FI': 4000, 'T-PT': 4000}
assert summary['checkpoint_ledger_count'] == 111
assert summary['model_selection_frozen_before_label_access']
assert summary['evaluation_labels_loaded']
